# IMDB Sentiment Analysis with GRU — Source Code

**Đề tài R&D**: Phân tích cảm xúc trên IMDB Reviews bằng kiến trúc **GRU (Gated Recurrent Unit)** trong PyTorch.

**Cách chạy**:
- **Google Colab**: chạy cell `Setup` đầu tiên — nó clone repo và `pip install -e .`.
- **Local Jupyter**: clone repo, `pip install -e ".[dev]"`, mở notebook tại repo root.

## Setup — môi trường + reproducibility

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PhongNguyenTrung/ml-imdb-gru.git"
REPO_DIR = "ml-imdb-gru"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Make src/ importable in both Colab and local Jupyter.
if "src" not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

print("cwd:", os.getcwd())
print("python:", sys.version.split()[0])

In [ ]:
import torch
from imdb_gru.utils import set_seed

SEED = 42
set_seed(SEED)
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
)
print(f"PyTorch {torch.__version__} | device={DEVICE}")

---
## Req 1 — Tải dữ liệu & EDA

Sử dụng thư viện `datasets` (HuggingFace) để tải IMDB Reviews (25 000 train + 25 000 test, balanced 50/50). Hiển thị **5 mẫu tiêu biểu** (balanced giữa pos/neg) và thống kê độ dài văn bản.

In [ ]:
from imdb_gru.data import IMDBLoader

loader = IMDBLoader(seed=SEED)
loader.summary()
samples = loader.show_samples(n=5, split="train", balanced=True)
loader.text_length_stats(split="train");

---
## Req 2 — Tiền xử lý + Vocabulary (10 000) + Padding/Truncating (max_len=256)

Pipeline: raw text → `RegexTokenizer` (strip HTML, lower-case, regex split) → `Vocabulary.encode` → pad/truncate.

**Quan trọng — Chống Data Leakage**: `Vocabulary.fit()` chỉ gọi trên **train split** (sau khi đã tách 10% làm validation). Val/test gặp token lạ → map thành `<unk>`.

In [ ]:
from imdb_gru.data import RegexTokenizer

tok = RegexTokenizer()
print("Raw text:")
print(" ", samples[0].text[:200], "...\n")
print("Tokens (after HTML strip + lowercase + regex split):")
print(" ", tok(samples[0].text)[:30])

In [ ]:
from imdb_gru.data import build_dataloaders

dm = build_dataloaders(
    loader.train, loader.test,
    val_ratio=0.1, max_len=256, vocab_size=10_000,
    batch_size=64, num_workers=2, seed=SEED,
)
print(f"Vocab size:   {len(dm.vocabulary)}")
print(f"Train batches: {len(dm.train_loader)}")
print(f"Val batches:   {len(dm.val_loader)}")
print(f"Test batches:  {len(dm.test_loader)}")

# Show one batch to verify shape.
batch = next(iter(dm.train_loader))
print(f"\nbatch['input_ids'].shape = {tuple(batch['input_ids'].shape)}  (B, T=256)")
print(f"batch['lengths'][:8]    = {batch['lengths'][:8].tolist()}")
print(f"batch['labels'][:8]     = {batch['labels'][:8].tolist()}")

---
## Req 3 — Kiến trúc mô hình: Embedding → GRU → Linear

**Toán học GRU** (tại bước thời gian $t$, hidden size $H$):

$$
\begin{aligned}
r_t &= \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}) & \text{reset gate} \\
z_t &= \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}) & \text{update gate} \\
n_t &= \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn})) & \text{candidate}\\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1} & \text{hidden}
\end{aligned}
$$

**Ý nghĩa số lượng neuron / param**:
- `Embedding(V=10000, E=128)` → $V \cdot E = 1{,}280{,}000$ params; mỗi token nhận vector dày 128 chiều.
- `GRU(E=128, H=128, 1 layer)` → $3(E H + H^2 + 2H) = 99{,}072$ params; $H=128$ là số neuron memory.
- `Linear(H, 1)` → $H+1 = 129$ params; 1 logit cho Bernoulli, đưa qua BCEWithLogits.

In [ ]:
from imdb_gru.models import GRUClassifier, GRUClassifierConfig

model_cfg = GRUClassifierConfig(
    vocab_size=len(dm.vocabulary),
    embed_dim=128, hidden_dim=128,
    num_layers=1, bidirectional=False,
    dropout=0.3,
)
model = GRUClassifier(model_cfg)
print(model)
print("\nTrainable parameters:")
for k, v in model.count_parameters().items():
    print(f"  {k:>12}: {v:>9,d}")

---
## Req 4 — Hàm mất mát + Bộ tối ưu

**Loss = `BCEWithLogitsLoss`**: cho bài toán binary, $y \mid x \sim \text{Bernoulli}(p)$, NLL là
$\mathcal{L} = -[y \log \sigma(z) + (1-y) \log(1 - \sigma(z))]$. PyTorch dùng dạng softplus ổn định số học, tránh `log(0)` khi $\sigma(z) \to 0$ hoặc $1$.

**Optimizer = Adam**:
$$ m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,\quad \theta_t = \theta_{t-1} - \eta\, \hat m_t / (\sqrt{\hat v_t} + \epsilon) $$
Step-size adaptive theo từng tham số $\Rightarrow$ rất phù hợp với gradient anisotropic của GRU (embedding sparse, recurrent dense, head nhỏ). Bias correction giảm noise BPTT trên sequence dài.

In [ ]:
from imdb_gru.training import build_loss, build_optimizer

loss_fn = build_loss("bce_with_logits")
optimizer = build_optimizer(
    model.parameters(),
    optimizer_name="adam",
    lr=1e-3,
    weight_decay=1e-5,   # ← L2 regularization (Req 6)
)
print(loss_fn)
print(optimizer)

---
## Req 5 + Req 6 — Vòng lặp huấn luyện + Regularization

**Req 5 (training loop)**: `Trainer` chạy train/val mỗi epoch, log Loss + Accuracy cả 2 split, lưu `history.json` + checkpoint best model.

**Req 6 (3 kỹ thuật chống quá khớp đồng thời)**:
1. **Dropout** $p=0.3$ — trên pooled hidden state trước Linear (random feature masking, equivalent với Bayesian model averaging).
2. **Weight Decay** $\lambda = 10^{-5}$ trong Adam — penalty $\lambda \|\theta\|_2^2$ kéo trọng số về 0.
3. **Early Stopping** — `patience=3` epoch nếu `val_loss` không cải thiện thì dừng (giới hạn ngầm capacity → giảm variance).

In [ ]:
from imdb_gru.training import Trainer, TrainerConfig

trainer_cfg = TrainerConfig(
    epochs=5,                       # demo: 5 epochs (Colab T4 ~3-5 phút)
    grad_clip=1.0,
    log_dir="artifacts/runs",
    experiment_name="baseline",
    monitor="val_loss",
    early_stopping_patience=3,
)
trainer = Trainer(model=model, loss_fn=loss_fn, optimizer=optimizer,
                  config=trainer_cfg, device=DEVICE)

history = trainer.fit(dm.train_loader, dm.val_loader)

---
## Req 7 — 3 thí nghiệm so sánh siêu tham số

| # | Variant | Mục tiêu |
|---|---|---|
| 1 | LR cao = $5 \times 10^{-3}$ | Kiểm tra hội tụ nhanh vs dao động |
| 2 | LR thấp = $2 \times 10^{-4}$ | Trade off tốc độ vs generalization |
| 3 | Batch size = 256 (gốc 64) | Gradient ít noise hơn, ít update hơn |

Để rút ngắn thời gian notebook chạy ~ 10–15 phút, mỗi thí nghiệm chạy 3 epochs.

In [ ]:
def run_experiment(name, *, lr=1e-3, batch_size=64, epochs=3, dropout=0.3, weight_decay=1e-5):
    set_seed(SEED)
    dm_exp = build_dataloaders(
        loader.train, loader.test,
        val_ratio=0.1, max_len=256, vocab_size=10_000,
        batch_size=batch_size, num_workers=2, seed=SEED,
    )
    m = GRUClassifier(GRUClassifierConfig(
        vocab_size=len(dm_exp.vocabulary),
        embed_dim=128, hidden_dim=128, dropout=dropout,
    ))
    opt = build_optimizer(m.parameters(), lr=lr, weight_decay=weight_decay)
    tr = Trainer(
        model=m, loss_fn=build_loss("bce_with_logits"), optimizer=opt,
        config=TrainerConfig(epochs=epochs, log_dir="artifacts/runs",
                             experiment_name=name, early_stopping_patience=None),
        device=DEVICE,
    )
    return tr.fit(dm_exp.train_loader, dm_exp.val_loader)

_ = run_experiment("exp_lr_high", lr=5e-3)
_ = run_experiment("exp_lr_low",  lr=2e-4)
_ = run_experiment("exp_bs_large", batch_size=256)

---
## Req 8 — Trực quan hoá Learning Curves

Vẽ đường cong Loss + Accuracy (Train vs Validation) cho baseline, rồi so sánh validation curves giữa 3 thí nghiệm Req 7.

In [ ]:
from imdb_gru.visualization import plot_learning_curves, plot_experiment_comparison

plot_learning_curves(
    "artifacts/runs/baseline/history.json",
    title="Baseline — Learning Curves",
    save_path="artifacts/figures/baseline_learning_curves.png",
    show=True,
);

In [ ]:
plot_experiment_comparison(
    histories=[
        ("LR=5e-3",   "artifacts/runs/exp_lr_high/history.json"),
        ("LR=2e-4",   "artifacts/runs/exp_lr_low/history.json"),
        ("BS=256",    "artifacts/runs/exp_bs_large/history.json"),
    ],
    save_path="artifacts/figures/experiment_comparison.png",
    show=True,
);

---
## Req 9 — Đánh giá chi tiết trên Test set

Confusion Matrix + `classification_report` (Precision, Recall, F1) trên 25 000 mẫu test, sau đó **phân tích các ca FP / FN** — in văn bản gốc của các sample mà model dự đoán sai với confidence cao nhất.

In [ ]:
import torch
from imdb_gru.evaluation import Evaluator, ErrorAnalyzer

# Load best checkpoint từ baseline run.
ckpt = torch.load("artifacts/runs/baseline/best.pt", map_location="cpu")
best_model = GRUClassifier(model_cfg)
best_model.load_state_dict(ckpt["state_dict"])

evaluator = Evaluator(best_model, device=DEVICE)
result = evaluator.evaluate(dm.test_loader)

print("=== Classification Report (Test) ===")
print(result.report)
print(f"Accuracy:  {result.accuracy:.4f}")
print(f"Precision: {result.precision:.4f}")
print(f"Recall:    {result.recall:.4f}")
print(f"F1-score:  {result.f1:.4f}")

Evaluator.plot_confusion_matrix(
    result, save_path="artifacts/figures/confusion_matrix.png", show=True,
);
Evaluator.plot_confusion_matrix(
    result, normalize=True,
    save_path="artifacts/figures/confusion_matrix_normalized.png", show=True,
);

In [ ]:
# Phân tích lỗi: in 5 FP + 5 FN với confidence cao nhất.
test_texts = list(loader.test["text"])
analyzer = ErrorAnalyzer(result, test_texts)
_ = analyzer.print_report(n_per_class=5, max_chars=400)

---
## Req 10 — Mở rộng lý thuyết: Self-Attention / Transformer

**Nhược điểm cố hữu của GRU**: cập nhật $h_t = (1-z_t) \odot n_t + z_t \odot h_{t-1}$ **bắt buộc tuần tự** trên $t$. Hai hậu quả:

1. Wall-clock forward pass có depth $\Theta(T)$ → không tận dụng được GPU parallelism.
2. Path-length từ token $i$ đến output là $T-i$ → long-range dependency vẫn bị suy giảm dù có gating.

**Self-Attention** (Vaswani et al., 2017) thay thế recurrence bằng **một bước pooling toàn cục song song**:

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

với $Q = XW_Q$, $K = XW_K$, $V = XW_V$. Hệ số $1/\sqrt{d_k}$ ngăn softmax bão hoà khi $d_k$ lớn.

**Multi-head**: $\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_H) W_O$ — mỗi head học một quan hệ ngữ nghĩa khác nhau.

**Lợi ích**:
- Sequential depth $O(T) \to O(1)$ (parallel).
- Direct $O(1)$ connectivity giữa mọi cặp token.
- Trade $O(Td^2)$ recurrent → $O(T^2 d)$ attention (chấp nhận được với $T=256$).

Sanity-check 2 model skeleton đã build trong `imdb_gru/models/attention_extension.py`:

In [ ]:
from imdb_gru.models import (
    GRUAttentionClassifier, TransformerEncoderClassifier, TransformerEncoderConfig,
)

# (a) GRU + Additive Self-Attention pooling.
gru_attn = GRUAttentionClassifier(model_cfg)
x = torch.randint(0, len(dm.vocabulary), (4, 256))
lengths = torch.tensor([256, 128, 64, 32])
out = gru_attn(x, lengths)
print(f"GRUAttentionClassifier:        out.shape={tuple(out.shape)}  (B,)")
print(f"  trainable params = {sum(p.numel() for p in gru_attn.parameters()):,}")

# (b) Pure Transformer Encoder.
tx_cfg = TransformerEncoderConfig(
    vocab_size=len(dm.vocabulary),
    embed_dim=128, num_heads=4, num_layers=2,
    feedforward_dim=256, dropout=0.1, max_len=256,
)
tx = TransformerEncoderClassifier(tx_cfg)
out_tx = tx(x, lengths)
print(f"TransformerEncoderClassifier:  out.shape={tuple(out_tx.shape)}  (B,)")
print(f"  trainable params = {sum(p.numel() for p in tx.parameters()):,}")

---
## Tổng kết

| Req | Nội dung | Trạng thái |
|---|---|---|
| 1 | Load IMDB + 5 mẫu + length stats | ✅ |
| 2 | RegexTokenizer + Vocabulary(10k, no leakage) + Dataset/DataLoader(max_len=256) | ✅ |
| 3 | Embedding → GRU → Linear + math + param count | ✅ |
| 4 | BCEWithLogitsLoss + Adam + rationale | ✅ |
| 5 | Train/val loop, log Loss+Acc mỗi epoch | ✅ |
| 6 | Dropout + Weight Decay + Early Stopping | ✅ |
| 7 | 3 thí nghiệm LR-high / LR-low / BS-large | ✅ |
| 8 | Learning curves + cross-experiment comparison | ✅ |
| 9 | Confusion matrix + Precision/Recall/F1 + FP/FN samples | ✅ |
| 10 | Self-Attention + Transformer Encoder (theory + working skeleton) | ✅ |

